# Caso H · 04 RAG documental — TF-IDF como sustituto ligero de embeddings

> _Tutorial · Caso de uso: **H — RAG + Chatbot** · Capa Medallion: **oro** · Spec: `docs/specs/synthetic-bms/01-product-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Implementar un RAG mínimo (TF-IDF + cosine) sobre los 12 docs CENTINELA+ / OMS / Medallion del repo. Sin LLM ni ElasticSearch externos.


## 2. Qué se aprende

- Tokenización + TF-IDF.
- Retrieval top-k.
- Cómo evaluar Recall@k con un golden set.


## 3. Contexto del caso de uso

TF-IDF demuestra el patrón sin necesitar GPU ni API keys. Cuando llegue Sentence-Transformers, basta con cambiar el vectorizador.


## 4. Relación con CENTINELA+

Mismo retriever; otro vectorizador en prod.


## 5. Relación con Medallion

Bronce: docs markdown; Plata: vectores; Oro: tool.


## 6. Datos de entrada

12 .md en `notebooks/_data/docs_rag_seed/`.


## 7. Schema CAPTIA esperado

No aplica.


## 8. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [ ]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


## 9. Carga de datos o mock

Cargamos los 12 docs.


In [ ]:
docs_dir = ROOT / "notebooks" / "_data" / "docs_rag_seed"
docs = []
for p in sorted(docs_dir.glob("*.md")):
    docs.append({"id": p.stem, "text": p.read_text(encoding="utf-8")})
df_docs = pd.DataFrame(docs)
print("docs:", len(df_docs))
df_docs.head(3)


## 10. Exploración paso a paso

TF-IDF + cosine.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vec = TfidfVectorizer(stop_words=None, ngram_range=(1, 2), min_df=1)
M = vec.fit_transform(df_docs["text"])

def retrieve(query: str, k: int = 3):
    qv = vec.transform([query])
    sims = cosine_similarity(qv, M)[0]
    order = np.argsort(-sims)[:k]
    return df_docs.iloc[order].assign(score=sims[order]).reset_index(drop=True)

print(retrieve("¿Qué es CENTINELA+?")[["id", "score"]])


## 11. Transformación bronce → plata

Vectorizamos.


## 12. Construcción de capa oro

La función retrieve es la tool RAG.


## 13. Visualizaciones explicativas

Heatmap docs × queries del golden set.


In [ ]:
gs = pd.read_csv(ROOT / "notebooks/_data/chatbot_golden_set.csv")
gs_rag = gs[gs["expected_mechanism"] == "rag"].head(8)
sims_matrix = cosine_similarity(vec.transform(gs_rag["question"]), M)
plt.figure(figsize=(8, 3))
plt.imshow(sims_matrix, aspect="auto", cmap="viridis")
plt.colorbar()
plt.yticks(range(len(gs_rag)), gs_rag["question"].str[:30] + "...")
plt.xticks(range(len(df_docs)), df_docs["id"], rotation=90, fontsize=8)
plt.title("Cosine similarity preguntas RAG vs docs")
plt.tight_layout()


## 14. Validaciones

Top-1 score > 0 para todas las preguntas RAG.


In [ ]:
for q in gs_rag["question"]:
    r = retrieve(q, k=1)
    assert r["score"].iloc[0] > 0
print("Retrieval OK")


## 15. Errores comunes

1. Usar TF-IDF sin n-gramas (frases multi-palabra fallan).
2. Olvidar lemmatización en español.
3. No filtrar duplicados antes de indexar.


## 16. Ejercicios propuestos

1. Añade un re-ranker BM25.
2. Sustituye TF-IDF por `sentence-transformers/multilingual-e5`.
3. Implementa eval Recall@5 sobre golden set.


## 17. Cómo se reutiliza con datos reales

Mismo retriever; producción usa Sentence-Transformers + ES.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `08_case_H_rag_chatbot/05_evaluacion_chatbot.ipynb`.
- Documento web del caso: `docs/use-cases/case-h-rag-chatbot.md`.
